In [1]:
import pandas as pd
tips = pd.read_csv("tips.csv")


In [6]:
# How many bills, average bill, average tip
print(tips.groupby("day")["total_bill"].size())
print(tips.groupby("day")["total_bill"].mean())
print(tips.groupby("day")["tip"].mean())


day
Fri     19
Sat     87
Sun     76
Thur    62
Name: total_bill, dtype: int64
day
Fri     17.151579
Sat     20.441379
Sun     21.410000
Thur    17.682742
Name: total_bill, dtype: float64
day
Fri     2.734737
Sat     2.993103
Sun     3.255132
Thur    2.771452
Name: tip, dtype: float64


In [14]:
# How many bills, average bill, average tip using:
# Named Aggregations
summary = tips.groupby("day").agg(
    Bill_count= ("total_bill",'size'),
    Avg_bill= ("total_bill",'mean'),
    Avg_tip= ("tip",'mean')
).round(2)
print(summary)

      Bill_count  Avg_bill  Avg_tip
day                                
Fri           19     17.15     2.73
Sat           87     20.44     2.99
Sun           76     21.41     3.26
Thur          62     17.68     2.77


In [11]:
# Bill_count, avg_bill, avg_party (same breakdown by time rather than by day: Lunch vs Dinner)
tips.groupby("time").agg(
    Bill_count = ("total_bill",'size'),
    Avg_bill= ("total_bill",'mean'),
    Avg_party= ("size",'mean')
).round(2)

,Bill_count,Avg_bill,Avg_party
time,,,
Dinner,176,20.80,2.63
Lunch,68,17.17,2.41


In [15]:
# Drop the days where the avg_tip is under 3
summary[summary["Avg_tip"]>=3]

,Bill_count,Avg_bill,Avg_tip
day,,,
Sun,76,21.41,3.26


In [ ]:
print(summary.columns) # Note that 'day' is not shown as a column
print(summary.index) # Note that the values of 'day' are shown as indexes


Index(['Bill_count', 'Avg_bill', 'Avg_tip'], dtype='str')
Index(['Fri', 'Sat', 'Sun', 'Thur'], dtype='str', name='day')


In [20]:
# reset_index : pushes the index back into a normal column and hands you a plain 0,1,2... index instead
flat = summary.reset_index()
print(flat.columns) # 'day' is included in columns now, instead of indexes
print(flat.head())

Index(['day', 'Bill_count', 'Avg_bill', 'Avg_tip'], dtype='str')
    day  Bill_count  Avg_bill  Avg_tip
0   Fri          19     17.15     2.73
1   Sat          87     20.44     2.99
2   Sun          76     21.41     3.26
3  Thur          62     17.68     2.77


In [ ]:
# Convert the file summary to csv format
summary.to_csv("Report.csv")
flat = pd.read_csv("Report.csv")
flat.head()

,day,Bill_count,Avg_bill,Avg_tip
0,Fri,19,17.15,2.73
1,Sat,87,20.44,2.99
2,Sun,76,21.41,3.26
3,Thur,62,17.68,2.77


In [26]:
# A smoker-section report: per smoker group- the no. of bills, average bill and average tip. Save it as "smoker_report.csv". 
smoker = tips.groupby("smoker").agg(
    Bill_count = ("total_bill",'size'),
    Avg_bill = ("total_bill",'mean'),
    Avg_tip = ("tip",'mean')
).round(2)
smoker.reset_index() # including smoker as a separate column instead of indexes
smoker.to_csv("Smoker_Report.csv")
smokerr = pd.read_csv("Smoker_Report.csv")
smokerr.head()

,smoker,Bill_count,Avg_bill,Avg_tip
0,No,151,19.19,2.99
1,Yes,93,20.76,3.01


In [ ]:
# idxmax() returns the index label of the row holding the maximum - not the max value, the row index.
print(tips.groupby("day")['tip'].idxmax())
print(tips.groupby("day")['tip'].max()) # max() gives the value of the maximum tip

day
Fri      95
Sat     170
Sun     183
Thur    141
Name: tip, dtype: int64
day
Fri      4.73
Sat     10.00
Sun      6.50
Thur     6.70
Name: tip, dtype: float64


In [ ]:
# The group on Saturday gave a tip of 10.00 pounds, who was that? Was it a big group?
print(tips.loc[170]) # Sat -> 170 (row index of max tip given on Sat) found by idxmax())

total_bill     50.81
tip             10.0
sex             Male
smoker           Yes
day              Sat
time          Dinner
size               3
Name: 170, dtype: object


In [42]:
# Which table on each day left the smallest tip relative to their bill?
# 1. Add a tip_pct column - tip as a percentage of the bill
# 2. Find the worst table on each day, showing the bill, the tip and the percentage.
# 3. Then run the same thing with idxmax and look hard at the Sunday row before you report it to her as your best customer.

tips["tip_pct"]= (tips["tip"]/tips["total_bill"]*100).round(2)
worst_tables = (tips.loc[
    tips.groupby("day")["tip_pct"].idxmin(),["day","total_bill","tip","tip_pct"]
])
print(worst_tables)
best_tables = (tips.loc[
    tips.groupby("day")["tip_pct"].idxmax(),["day", "total_bill", "tip", "tip_pct"]]
)
print(best_tables)


      day  total_bill   tip  tip_pct
90    Fri       28.97  3.00    10.36
237   Sat       32.83  1.17     3.56
0     Sun       16.99  1.01     5.94
146  Thur       18.64  1.36     7.30
      day  total_bill   tip  tip_pct
93    Fri       16.32  4.30    26.35
67    Sat        3.07  1.00    32.57
172   Sun        7.25  5.15    71.03
149  Thur        7.51  2.00    26.63
